# 04 — Multi-Seed Synthetic Data Generation

## Objective

Generate synthetic student engagement datasets using CTGAN
and TVAE under different missingness restoration strategies.

The experiment evaluates the stability of synthetic data
generation across multiple random seeds.

## Generators

1. CTGAN
2. TVAE

## Experimental strategies

1. Random missingness restoration with Engagement_Level.
2. From-column missingness restoration with Engagement_Level.

## Random seeds

- 42
- 123
- 2026
- 3407

## Experimental design

Each generator is independently trained for every seed
and experimental configuration.

The generated datasets are saved separately to preserve
reproducibility and prevent accidental overwriting.

## Missingness restoration

Random strategy:
Missing values are restored using feature-specific missingness
rates calculated from the real training data.

From-column strategy:
Generated missingness indicators determine which feature
values are restored as NaN.

## Data leakage prevention

- Only real training data is used for generator training.
- The real test dataset is never loaded.
- The test set is reserved for downstream evaluation.
- Engagement_Level is never masked.

## Generator settings

- Epochs: 300
- Generator configurations: CTGAN and TVAE
- Synthetic sample size: 21,328 rows per experiment

In [1]:
import json
import time
import random

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import sdv

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer

Experiment configuration

In [2]:
# -------------------------------
# Experiment configuration
# -------------------------------

FEATURES = [
    "nevents",
    "ndays_act",
    "ncontent",
    "nforum_posts",
]

FLAGS = [
    "nevents_missing",
    "ndays_act_missing",
    "nforum_posts_missing",
]

TARGET = "Engagement_Level"
TABLE = "students"

SEEDS = [42, 123, 2026, 3407]

EPOCHS = 300
N_TOTAL = 21328

GENERATORS = {
    "ctgan": {
        "class": CTGANSynthesizer,
        "kwargs": {
            "verbose": True,
            "log_frequency": False,
        },
    },
    "tvae": {
        "class": TVAESynthesizer,
        "kwargs": {},
    },
}

print("Seeds:", SEEDS)
print("Epochs:", EPOCHS)
print("Synthetic rows per experiment:", N_TOTAL)
print("Generators:", list(GENERATORS.keys()))

Seeds: [42, 123, 2026, 3407]
Epochs: 300
Synthetic rows per experiment: 21328
Generators: ['ctgan', 'tvae']


Reproducible seed function

In [3]:
def set_seeds(seed):
    """
    Set random seeds for Python, NumPy and PyTorch.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

verify the input datasets

In [5]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nCurrent directory contents:")
for item in Path.cwd().iterdir():
    print(item.name)

print("\nChecking likely project locations:")
for path in [
    Path.cwd() / "data" / "processed",
    Path.cwd().parent / "data" / "processed",
]:
    print(f"{path} -> {'FOUND' if path.exists() else 'NOT FOUND'}")

Current working directory:
d:\student_engagement_twin\notebooks

Current directory contents:
00_api_checks.ipynb
01_data.ipynb
02_preprocessing.ipynb
03_labeler.ipynb
04_generators.ipynb
05_fidelity.ipynb
06_classification.ipynb
07_explainability.ipynb

Checking likely project locations:
d:\student_engagement_twin\notebooks\data\processed -> NOT FOUND
d:\student_engagement_twin\data\processed -> FOUND


set the project paths

In [6]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "artifacts" / "robustness"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

required_files = [
    DATA_DIR / "train.csv",
    DATA_DIR / "gen_input_random_target.csv",
    DATA_DIR / "gen_input_from_column_target.csv",
]

for file in required_files:
    print(f"{file.name}: {'FOUND' if file.exists() else 'MISSING'}")

Project directory: d:\student_engagement_twin
Data directory: d:\student_engagement_twin\data\processed
Output directory: d:\student_engagement_twin\artifacts\robustness
train.csv: FOUND
gen_input_random_target.csv: FOUND
gen_input_from_column_target.csv: FOUND


inspect the input datasets

In [7]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
random_df = pd.read_csv(DATA_DIR / "gen_input_random_target.csv")
column_df = pd.read_csv(DATA_DIR / "gen_input_from_column_target.csv")

print("TRAIN DATA")
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

print("\nRANDOM TARGET INPUT")
print("Shape:", random_df.shape)
print("Columns:", random_df.columns.tolist())

print("\nFROM-COLUMN TARGET INPUT")
print("Shape:", column_df.shape)
print("Columns:", column_df.columns.tolist())

print("\nTARGET DISTRIBUTION")
print(train_df[TARGET].value_counts(normalize=True).sort_index())

print("\nMISSING VALUES IN GENERATOR INPUTS")
print("Random input:")
print(random_df.isna().sum())

print("\nFrom-column input:")
print(column_df.isna().sum())

TRAIN DATA
Shape: (21328, 5)
Columns: ['nevents', 'ndays_act', 'ncontent', 'nforum_posts', 'Engagement_Level']

RANDOM TARGET INPUT
Shape: (21328, 5)
Columns: ['nevents', 'ndays_act', 'ncontent', 'nforum_posts', 'Engagement_Level']

FROM-COLUMN TARGET INPUT
Shape: (21328, 8)
Columns: ['nevents', 'ndays_act', 'ncontent', 'nforum_posts', 'nevents_missing', 'ndays_act_missing', 'nforum_posts_missing', 'Engagement_Level']

TARGET DISTRIBUTION
Engagement_Level
0    0.824597
1    0.175403
Name: proportion, dtype: float64

MISSING VALUES IN GENERATOR INPUTS
Random input:
nevents             0
ndays_act           0
ncontent            0
nforum_posts        0
Engagement_Level    0
dtype: int64

From-column input:
nevents                 0
ndays_act               0
ncontent                0
nforum_posts            0
nevents_missing         0
ndays_act_missing       0
nforum_posts_missing    0
Engagement_Level        0
dtype: int64


calculate the original missingness rates

In [8]:
MISS_RATES = (
    train_df[FEATURES]
    .isna()
    .mean()
    .to_dict()
)

print("Original training missingness rates:")
for feature, rate in MISS_RATES.items():
    print(f"{feature}: {rate:.4%}")

print("\nFrom-column missingness flag rates:")
for flag in FLAGS:
    rate = column_df[flag].mean()
    print(f"{flag}: {rate:.4%}")

print("\nChecking missingness flags:")
for flag in FLAGS:
    values = set(column_df[flag].unique())
    print(f"{flag}: {values}")

Original training missingness rates:
nevents: 17.0808%
ndays_act: 10.5308%
ncontent: 0.0000%
nforum_posts: 68.1077%

From-column missingness flag rates:
nevents_missing: 17.0808%
ndays_act_missing: 10.5308%
nforum_posts_missing: 68.1077%

Checking missingness flags:
nevents_missing: {np.int64(0), np.int64(1)}
ndays_act_missing: {np.int64(0), np.int64(1)}
nforum_posts_missing: {np.int64(0), np.int64(1)}


prepare generator metadata

In [9]:
def build_metadata(df, categorical_columns):
    metadata = Metadata.detect_from_dataframes(
        {TABLE: df}
    )

    for column in categorical_columns:
        metadata.update_column(
            column_name=column,
            table_name=TABLE,
            sdtype="categorical"
        )

    return metadata


random_metadata = build_metadata(
    random_df,
    categorical_columns=[TARGET]
)

column_metadata = build_metadata(
    column_df,
    categorical_columns=FLAGS + [TARGET]
)

print("Random-target metadata:")
print(random_metadata)

print("\nFrom-column-target metadata:")
print(column_metadata)

Random-target metadata:
{
    "tables": {
        "students": {
            "columns": {
                "nevents": {
                    "sdtype": "numerical",
                    "range_is_nullable": false,
                    "range_min": 1,
                    "range_max": 168896,
                    "decimal_places": 0
                },
                "ndays_act": {
                    "sdtype": "numerical",
                    "range_is_nullable": false,
                    "range_min": 1,
                    "range_max": 103,
                    "decimal_places": 0
                },
                "ncontent": {
                    "sdtype": "numerical",
                    "range_is_nullable": false,
                    "range_min": 4,
                    "range_max": 100,
                    "decimal_places": 0
                },
                "nforum_posts": {
                    "sdtype": "numerical",
                    "range_is_nullable": false,
                    "

verify column types

In [10]:
print("RANDOM-TARGET COLUMN TYPES")
for column, details in random_metadata.to_dict()["tables"][TABLE]["columns"].items():
    print(f"{column}: {details['sdtype']}")

print("\nFROM-COLUMN-TARGET COLUMN TYPES")
for column, details in column_metadata.to_dict()["tables"][TABLE]["columns"].items():
    print(f"{column}: {details['sdtype']}")

RANDOM-TARGET COLUMN TYPES
nevents: numerical
ndays_act: numerical
ncontent: numerical
nforum_posts: numerical
Engagement_Level: categorical

FROM-COLUMN-TARGET COLUMN TYPES
nevents: numerical
ndays_act: numerical
ncontent: numerical
nforum_posts: numerical
nevents_missing: categorical
ndays_act_missing: categorical
nforum_posts_missing: categorical
Engagement_Level: categorical


create the output folders

In [11]:
ADDITIONAL_SEEDS = [123, 2026, 3407]

RESTORATION_SEED = 42

for seed in ADDITIONAL_SEEDS:
    seed_dir = OUTPUT_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    print(f"Seed {seed}: {seed_dir}")

print("\nExisting seed-42 results will not be modified.")
print("Additional training runs:", len(ADDITIONAL_SEEDS) * 2 * 2)

Seed 123: d:\student_engagement_twin\artifacts\robustness\seed_123
Seed 2026: d:\student_engagement_twin\artifacts\robustness\seed_2026
Seed 3407: d:\student_engagement_twin\artifacts\robustness\seed_3407

Existing seed-42 results will not be modified.
Additional training runs: 12


create restoration functions

In [12]:
def restore_random(df, miss_rates, seed):
    """Restore missing values using fixed feature-specific rates."""

    required = FEATURES + [TARGET]
    missing_columns = set(required) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Random strategy is missing columns: {missing_columns}"
        )

    restored = df[required].copy()
    rng = np.random.default_rng(seed)

    for feature in FEATURES:
        rate = miss_rates[feature]

        if not 0 <= rate <= 1:
            raise ValueError(f"Invalid missingness rate for {feature}: {rate}")

        mask = rng.random(len(restored)) < rate
        restored.loc[mask, feature] = np.nan

    if restored[TARGET].isna().any():
        raise ValueError("Engagement_Level must never be missing.")

    return restored


def restore_from_flags(df):
    """Restore missing values using generated missingness indicators."""

    required = FEATURES + FLAGS + [TARGET]
    missing_columns = set(required) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"From-column strategy is missing columns: {missing_columns}"
        )

    restored = df[required].copy()

    for flag in FLAGS:
        values = set(restored[flag].dropna().unique())

        if not values.issubset({0, 1}):
            raise ValueError(
                f"{flag} contains invalid values: {values}"
            )

        if restored[flag].isna().any():
            raise ValueError(f"{flag} contains missing values.")

    for feature in FEATURES:
        flag = f"{feature}_missing"

        if flag in FLAGS:
            mask = restored[flag] == 1
            restored.loc[mask, feature] = np.nan

    restored = restored[FEATURES + [TARGET]]

    if restored[TARGET].isna().any():
        raise ValueError("Engagement_Level must never be missing.")

    return restored

Test the functions without training

In [13]:
test_random = restore_random(
    random_df.head(1000),
    MISS_RATES,
    seed=RESTORATION_SEED
)

test_column = restore_from_flags(
    column_df.head(1000)
)

print("Random restoration test:")
print(test_random.isna().sum())

print("\nFrom-column restoration test:")
print(test_column.isna().sum())

print("\nOutput columns:")
print("Random:", test_random.columns.tolist())
print("From-column:", test_column.columns.tolist())

print("\nTarget missing values:")
print("Random:", test_random[TARGET].isna().sum())
print("From-column:", test_column[TARGET].isna().sum())

Random restoration test:
nevents             190
ndays_act            94
ncontent              0
nforum_posts        687
Engagement_Level      0
dtype: int64

From-column restoration test:
nevents             172
ndays_act            83
ncontent              0
nforum_posts        667
Engagement_Level      0
dtype: int64

Output columns:
Random: ['nevents', 'ndays_act', 'ncontent', 'nforum_posts', 'Engagement_Level']
From-column: ['nevents', 'ndays_act', 'ncontent', 'nforum_posts', 'Engagement_Level']

Target missing values:
Random: 0
From-column: 0


training function

In [17]:

def train_generator(
    tag,
    generator_class,
    generator_kwargs,
    input_df,
    metadata,
    seed,
    strategy,
):
    """
    Train one SDV generator, generate synthetic records,
    restore missing values, and save all outputs.
    """

    # 1. Create a separate output folder for this seed
    seed_dir = OUTPUT_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    csv_path = seed_dir / f"{tag}_restored.csv"
    model_path = seed_dir / f"{tag}.pkl"

    # 2. Prevent accidental overwriting
    if csv_path.exists() or model_path.exists():
        raise FileExistsError(
            f"Output already exists for {tag}, seed {seed}.\n"
            f"CSV: {csv_path}\n"
            f"Model: {model_path}\n"
            "Inspect existing files before rerunning."
        )

    print("\n" + "=" * 60)
    print(f"Experiment: {tag}")
    print(f"Seed: {seed}")
    print(f"Strategy: {strategy}")
    print("=" * 60)

    # 3. Set random seeds
    set_seeds(seed)

    # 4. Initialize the SDV synthesizer
    synthesizer = generator_class(
        metadata,
        epochs=EPOCHS,
        **generator_kwargs,
    )

    # 5. Train the synthesizer
    start_time = time.time()

    synthesizer.fit({TABLE: input_df})

    training_seconds = time.time() - start_time

    print(
        f"\nTraining completed in "
        f"{training_seconds / 60:.2f} minutes."
    )

    # 6. Generate synthetic records
    set_seeds(seed)

    synthetic_result = synthesizer.sample(
        num_rows=N_TOTAL
    )

    # SDV may return a dictionary containing the table DataFrame.
    if isinstance(synthetic_result, dict):
        if TABLE not in synthetic_result:
            raise ValueError(
                f"SDV output does not contain table '{TABLE}'."
            )
        raw_synthetic = synthetic_result[TABLE]
    else:
        raw_synthetic = synthetic_result

    if not isinstance(raw_synthetic, pd.DataFrame):
        raise TypeError("Synthetic output is not a pandas DataFrame.")

    # 7. Restore missing values
    if strategy == "random":

        restored = restore_random(
            raw_synthetic,
            MISS_RATES,
            seed=RESTORATION_SEED,
        )

    elif strategy == "from_column":

        restored = restore_from_flags(raw_synthetic)

    else:
        raise ValueError(
            f"Unknown restoration strategy: {strategy}"
        )

    # 8. Validate the final synthetic dataset
    expected_columns = FEATURES + [TARGET]

    if restored.shape != (N_TOTAL, len(expected_columns)):
        raise ValueError(
            f"Unexpected output shape: {restored.shape}"
        )

    if restored.columns.tolist() != expected_columns:
        raise ValueError(
            f"Unexpected columns: {restored.columns.tolist()}"
        )

    if restored[TARGET].isna().any():
        raise ValueError(
            "Generated Engagement_Level contains missing values."
        )

    if not set(restored[TARGET].unique()).issubset({0, 1}):
        raise ValueError(
            "Generated Engagement_Level contains invalid labels."
        )

    # 9. Save the synthetic dataset and trained model
    restored.to_csv(csv_path, index=False)

    synthesizer.save(filepath=str(model_path))

    elapsed = time.time() - start_time

    # 10. Record experiment information
    result = {
        "experiment": tag,
        "seed": seed,
        "strategy": strategy,
        "generator": tag.split("_")[0],
        "epochs": EPOCHS,
        "synthetic_rows": len(restored),
        "training_seconds": round(training_seconds, 2),
        "total_seconds": round(elapsed, 2),
        "csv_path": str(csv_path),
        "model_path": str(model_path),
        "missing_values": {
            key: int(value)
            for key, value in restored.isna().sum().items()
        },
        "class_distribution": {
            str(key): float(value)
            for key, value in (
                restored[TARGET]
                .value_counts(normalize=True)
                .sort_index()
                .items()
            )
        },
    }

    print("\nExperiment completed successfully.")
    print(f"Training time: {training_seconds / 60:.2f} minutes")
    print(f"Total time: {elapsed / 60:.2f} minutes")
    print(f"Dataset saved: {csv_path}")
    print(f"Model saved: {model_path}")

    return result


prepare the experiment configurations

In [18]:
EXPERIMENTS = {
    "random_target": {
        "input": random_df,
        "metadata": random_metadata,
        "strategy": "random",
    },
    "from_column_target": {
        "input": column_df,
        "metadata": column_metadata,
        "strategy": "from_column",
    },
}

print("Planned additional experiments:")

for seed in ADDITIONAL_SEEDS:
    for variant, config in EXPERIMENTS.items():
        for generator_name in GENERATORS:
            print(
                f"Seed {seed} | "
                f"{generator_name.upper()} | "
                f"{variant}"
            )

print("\nTotal training runs:", 
      len(ADDITIONAL_SEEDS) * len(EXPERIMENTS) * len(GENERATORS))

Planned additional experiments:
Seed 123 | CTGAN | random_target
Seed 123 | TVAE | random_target
Seed 123 | CTGAN | from_column_target
Seed 123 | TVAE | from_column_target
Seed 2026 | CTGAN | random_target
Seed 2026 | TVAE | random_target
Seed 2026 | CTGAN | from_column_target
Seed 2026 | TVAE | from_column_target
Seed 3407 | CTGAN | random_target
Seed 3407 | TVAE | random_target
Seed 3407 | CTGAN | from_column_target
Seed 3407 | TVAE | from_column_target

Total training runs: 12


In [19]:
first_result = train_generator(
    tag="ctgan_random_target",
    generator_class=GENERATORS["ctgan"]["class"],
    generator_kwargs=GENERATORS["ctgan"]["kwargs"],
    input_df=EXPERIMENTS["random_target"]["input"],
    metadata=EXPERIMENTS["random_target"]["metadata"],
    seed=123,
    strategy="random",
)


Experiment: ctgan_random_target
Seed: 123
Strategy: random


Gen. (-00.77) | Discrim. (-00.40): 100%|██████████| 300/300 [06:29<00:00,  1.30s/it]



Training completed in 6.71 minutes.

Experiment completed successfully.
Training time: 6.71 minutes
Total time: 6.71 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_123\ctgan_random_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_123\ctgan_random_target.pkl


In [20]:
generated_df = pd.read_csv(
    OUTPUT_DIR / "seed_123" / "ctgan_random_target_restored.csv"
)

print("Shape:", generated_df.shape)

print("\nMissing values:")
print(generated_df.isna().sum())

print("\nTarget distribution:")
print(generated_df[TARGET].value_counts())

print("\nData types:")
print(generated_df.dtypes)

print("\nFirst five rows:")
display(generated_df.head())

Shape: (21328, 5)

Missing values:
nevents              3652
ndays_act            2236
ncontent                0
nforum_posts        14606
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17574
1     3754
Name: count, dtype: int64

Data types:
nevents             float64
ndays_act           float64
ncontent            float64
nforum_posts        float64
Engagement_Level      int64
dtype: object

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,89.0,3.0,11.0,2.0,0
1,296.0,2.0,100.0,1.0,0
2,1.0,2.0,15.0,NaN,0
3,1729.0,NaN,37.0,1.0,0
4,NaN,2.0,73.0,NaN,0


TVAE, Seed 123

In [21]:
tvae_random_result = train_generator(
    tag="tvae_random_target",
    generator_class=GENERATORS["tvae"]["class"],
    generator_kwargs=GENERATORS["tvae"]["kwargs"],
    input_df=EXPERIMENTS["random_target"]["input"],
    metadata=EXPERIMENTS["random_target"]["metadata"],
    seed=123,
    strategy="random",
)


Experiment: tvae_random_target
Seed: 123
Strategy: random

Training completed in 2.76 minutes.

Experiment completed successfully.
Training time: 2.76 minutes
Total time: 2.76 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_123\tvae_random_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_123\tvae_random_target.pkl


validate TVAE output

In [22]:
tvae_df = pd.read_csv(
    OUTPUT_DIR / "seed_123" / "tvae_random_target_restored.csv"
)

print("Shape:", tvae_df.shape)

print("\nMissing values:")
print(tvae_df.isna().sum())

print("\nTarget distribution:")
print(tvae_df[TARGET].value_counts())

print("\nData types:")
print(tvae_df.dtypes)

print("\nFirst five rows:")
display(tvae_df.head())

Shape: (21328, 5)

Missing values:
nevents              3652
ndays_act            2236
ncontent                0
nforum_posts        14606
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17651
1     3677
Name: count, dtype: int64

Data types:
nevents             float64
ndays_act           float64
ncontent            float64
nforum_posts        float64
Engagement_Level      int64
dtype: object

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,78.0,3.0,23.0,1.0,0
1,288.0,6.0,40.0,2.0,0
2,446.0,7.0,100.0,NaN,0
3,2828.0,NaN,100.0,6.0,1
4,NaN,8.0,41.0,NaN,0


CTGAN, From-column, Seed 123

In [23]:
ctgan_column_result = train_generator(
    tag="ctgan_from_column_target",
    generator_class=GENERATORS["ctgan"]["class"],
    generator_kwargs=GENERATORS["ctgan"]["kwargs"],
    input_df=EXPERIMENTS["from_column_target"]["input"],
    metadata=EXPERIMENTS["from_column_target"]["metadata"],
    seed=123,
    strategy="from_column",
)


Experiment: ctgan_from_column_target
Seed: 123
Strategy: from_column


d:\student_engagement_twin\.venv\Lib\site-packages\sdv\single_table\base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-01.11) | Discrim. (+00.32): 100%|██████████| 300/300 [06:05<00:00,  1.22s/it]



Training completed in 6.30 minutes.

Experiment completed successfully.
Training time: 6.30 minutes
Total time: 6.30 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_123\ctgan_from_column_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_123\ctgan_from_column_target.pkl


validate CTGAN From-column output

In [24]:
ctgan_column_df = pd.read_csv(
    OUTPUT_DIR / "seed_123" / "ctgan_from_column_target_restored.csv"
)

print("Shape:", ctgan_column_df.shape)

print("\nMissing values:")
print(ctgan_column_df.isna().sum())

print("\nTarget distribution:")
print(ctgan_column_df[TARGET].value_counts())

print("\nData types:")
print(ctgan_column_df.dtypes)

print("\nFirst five rows:")
display(ctgan_column_df.head())

Shape: (21328, 5)

Missing values:
nevents              4029
ndays_act            3503
ncontent                0
nforum_posts        15123
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17851
1     3477
Name: count, dtype: int64

Data types:
nevents             float64
ndays_act           float64
ncontent              int64
nforum_posts        float64
Engagement_Level      int64
dtype: object

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,83.0,3.0,100,NaN,0
1,NaN,NaN,100,1.0,0
2,447.0,23.0,100,13.0,0
3,81.0,NaN,9,1.0,0
4,85.0,1.0,37,NaN,0


 Final experiment for Seed 123 — TVAE From-column

In [25]:
tvae_column_result = train_generator(
    tag="tvae_from_column_target",
    generator_class=GENERATORS["tvae"]["class"],
    generator_kwargs=GENERATORS["tvae"]["kwargs"],
    input_df=EXPERIMENTS["from_column_target"]["input"],
    metadata=EXPERIMENTS["from_column_target"]["metadata"],
    seed=123,
    strategy="from_column",
)


Experiment: tvae_from_column_target
Seed: 123
Strategy: from_column

Training completed in 2.30 minutes.

Experiment completed successfully.
Training time: 2.30 minutes
Total time: 2.30 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_123\tvae_from_column_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_123\tvae_from_column_target.pkl


Validate the final TVAE dataset

In [26]:
tvae_column_df = pd.read_csv(
    OUTPUT_DIR / "seed_123" / "tvae_from_column_target_restored.csv"
)

print("Shape:", tvae_column_df.shape)

print("\nMissing values:")
print(tvae_column_df.isna().sum())

print("\nTarget distribution:")
print(tvae_column_df[TARGET].value_counts())

print("\nData types:")
print(tvae_column_df.dtypes)

print("\nFirst five rows:")
display(tvae_column_df.head())

Shape: (21328, 5)

Missing values:
nevents              3417
ndays_act             356
ncontent                0
nforum_posts        14794
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17919
1     3409
Name: count, dtype: int64

Data types:
nevents             float64
ndays_act           float64
ncontent              int64
nforum_posts        float64
Engagement_Level      int64
dtype: object

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,187.0,5.0,42,NaN,0
1,1323.0,3.0,100,NaN,0
2,44.0,13.0,100,8.0,0
3,551.0,6.0,93,7.0,0
4,79.0,8.0,81,NaN,0


Start seed 2026 — CTGAN Random

In [27]:
ctgan_random_2026_result = train_generator(
    tag="ctgan_random_target",
    generator_class=GENERATORS["ctgan"]["class"],
    generator_kwargs=GENERATORS["ctgan"]["kwargs"],
    input_df=EXPERIMENTS["random_target"]["input"],
    metadata=EXPERIMENTS["random_target"]["metadata"],
    seed=2026,
    strategy="random",
)


Experiment: ctgan_random_target
Seed: 2026
Strategy: random


Gen. (-01.10) | Discrim. (+00.02): 100%|██████████| 300/300 [05:31<00:00,  1.11s/it]



Training completed in 5.75 minutes.

Experiment completed successfully.
Training time: 5.75 minutes
Total time: 5.75 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\ctgan_random_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\ctgan_random_target.pkl


validate the generated dataset


In [28]:
ctgan_random_2026_df = pd.read_csv(
    OUTPUT_DIR / "seed_2026" / "ctgan_random_target_restored.csv"
)

print("Shape:", ctgan_random_2026_df.shape)

print("\nMissing values:")
print(ctgan_random_2026_df.isna().sum())

print("\nTarget distribution:")
print(ctgan_random_2026_df[TARGET].value_counts())

print("\nFirst five rows:")
display(ctgan_random_2026_df.head())

Shape: (21328, 5)

Missing values:
nevents              3652
ndays_act            2236
ncontent                0
nforum_posts        14606
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17578
1     3750
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,32.0,5.0,23.0,3.0,0
1,68.0,14.0,100.0,4.0,0
2,103.0,3.0,39.0,NaN,0
3,190.0,NaN,23.0,1.0,0
4,NaN,3.0,100.0,NaN,0


TVAE, Seed 2026, Random

In [29]:
tvae_random_2026_result = train_generator(
    tag="tvae_random_target",
    generator_class=GENERATORS["tvae"]["class"],
    generator_kwargs=GENERATORS["tvae"]["kwargs"],
    input_df=EXPERIMENTS["random_target"]["input"],
    metadata=EXPERIMENTS["random_target"]["metadata"],
    seed=2026,
    strategy="random",
)


Experiment: tvae_random_target
Seed: 2026
Strategy: random

Training completed in 2.25 minutes.

Experiment completed successfully.
Training time: 2.25 minutes
Total time: 2.26 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\tvae_random_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\tvae_random_target.pkl


validate the dataset

In [30]:
tvae_random_2026_df = pd.read_csv(
    OUTPUT_DIR / "seed_2026" / "tvae_random_target_restored.csv"
)

print("Shape:", tvae_random_2026_df.shape)

print("\nMissing values:")
print(tvae_random_2026_df.isna().sum())

print("\nTarget distribution:")
print(tvae_random_2026_df[TARGET].value_counts())

print("\nFirst five rows:")
display(tvae_random_2026_df.head())

Shape: (21328, 5)

Missing values:
nevents              3652
ndays_act            2236
ncontent                0
nforum_posts        14606
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17213
1     4115
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,1495.0,8.0,100.0,7.0,0
1,398.0,7.0,90.0,2.0,1
2,123.0,9.0,38.0,NaN,0
3,5637.0,NaN,100.0,2.0,1
4,NaN,6.0,72.0,NaN,0


CTGAN, Seed 2026, From-column

In [31]:
ctgan_column_2026_result = train_generator(
    tag="ctgan_from_column_target",
    generator_class=GENERATORS["ctgan"]["class"],
    generator_kwargs=GENERATORS["ctgan"]["kwargs"],
    input_df=EXPERIMENTS["from_column_target"]["input"],
    metadata=EXPERIMENTS["from_column_target"]["metadata"],
    seed=2026,
    strategy="from_column",
)


Experiment: ctgan_from_column_target
Seed: 2026
Strategy: from_column


Gen. (-00.90) | Discrim. (-00.16): 100%|██████████| 300/300 [07:43<00:00,  1.55s/it]



Training completed in 8.05 minutes.

Experiment completed successfully.
Training time: 8.05 minutes
Total time: 8.06 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\ctgan_from_column_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\ctgan_from_column_target.pkl


validate CTGAN output

In [32]:
ctgan_column_2026_df = pd.read_csv(
    OUTPUT_DIR / "seed_2026" / "ctgan_from_column_target_restored.csv"
)

print("Shape:", ctgan_column_2026_df.shape)

print("\nMissing values:")
print(ctgan_column_2026_df.isna().sum())

print("\nTarget distribution:")
print(ctgan_column_2026_df[TARGET].value_counts())

print("\nFirst five rows:")
display(ctgan_column_2026_df.head())

Shape: (21328, 5)

Missing values:
nevents              2633
ndays_act            1546
ncontent                0
nforum_posts        15908
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    18445
1     2883
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,1.0,1.0,19,NaN,0
1,NaN,NaN,18,NaN,0
2,1274.0,5.0,100,16.0,0
3,128.0,9.0,19,NaN,0
4,194.0,NaN,99,NaN,0


TVAE, Seed 2026, From-column

In [33]:
tvae_column_2026_result = train_generator(
    tag="tvae_from_column_target",
    generator_class=GENERATORS["tvae"]["class"],
    generator_kwargs=GENERATORS["tvae"]["kwargs"],
    input_df=EXPERIMENTS["from_column_target"]["input"],
    metadata=EXPERIMENTS["from_column_target"]["metadata"],
    seed=2026,
    strategy="from_column",
)


Experiment: tvae_from_column_target
Seed: 2026
Strategy: from_column

Training completed in 2.05 minutes.

Experiment completed successfully.
Training time: 2.05 minutes
Total time: 2.05 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\tvae_from_column_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_2026\tvae_from_column_target.pkl


Validate the dataset

In [34]:
tvae_column_2026_df = pd.read_csv(
    OUTPUT_DIR / "seed_2026" / "tvae_from_column_target_restored.csv"
)

print("Shape:", tvae_column_2026_df.shape)

print("\nMissing values:")
print(tvae_column_2026_df.isna().sum())

print("\nTarget distribution:")
print(tvae_column_2026_df[TARGET].value_counts())

print("\nFirst five rows:")
display(tvae_column_2026_df.head())

Shape: (21328, 5)

Missing values:
nevents              2053
ndays_act               3
ncontent                0
nforum_posts        14013
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17237
1     4091
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,725.0,4.0,79,7.0,1
1,NaN,1.0,36,NaN,0
2,742.0,1.0,100,2.0,1
3,75.0,4.0,100,NaN,0
4,475.0,1.0,100,NaN,0


Seed 3407, CTGAN Random

In [35]:
ctgan_random_3407_result = train_generator(
    tag="ctgan_random_target",
    generator_class=GENERATORS["ctgan"]["class"],
    generator_kwargs=GENERATORS["ctgan"]["kwargs"],
    input_df=EXPERIMENTS["random_target"]["input"],
    metadata=EXPERIMENTS["random_target"]["metadata"],
    seed=3407,
    strategy="random",
)


Experiment: ctgan_random_target
Seed: 3407
Strategy: random


Gen. (-00.43) | Discrim. (+00.01): 100%|██████████| 300/300 [05:32<00:00,  1.11s/it]



Training completed in 5.78 minutes.

Experiment completed successfully.
Training time: 5.78 minutes
Total time: 5.78 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\ctgan_random_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\ctgan_random_target.pkl


validate the dataset

In [36]:
ctgan_random_3407_df = pd.read_csv(
    OUTPUT_DIR / "seed_3407" / "ctgan_random_target_restored.csv"
)

print("Shape:", ctgan_random_3407_df.shape)

print("\nMissing values:")
print(ctgan_random_3407_df.isna().sum())

print("\nTarget distribution:")
print(ctgan_random_3407_df[TARGET].value_counts())

print("\nFirst five rows:")
display(ctgan_random_3407_df.head())

Shape: (21328, 5)

Missing values:
nevents              3652
ndays_act            2236
ncontent                0
nforum_posts        14606
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17573
1     3755
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,1.0,2.0,22.0,3.0,0
1,439.0,4.0,20.0,1.0,0
2,122.0,2.0,13.0,NaN,0
3,251.0,NaN,100.0,1.0,0
4,NaN,6.0,100.0,NaN,0


TVAE, Seed 3407, Random

In [37]:
tvae_random_3407_result = train_generator(
    tag="tvae_random_target",
    generator_class=GENERATORS["tvae"]["class"],
    generator_kwargs=GENERATORS["tvae"]["kwargs"],
    input_df=EXPERIMENTS["random_target"]["input"],
    metadata=EXPERIMENTS["random_target"]["metadata"],
    seed=3407,
    strategy="random",
)


Experiment: tvae_random_target
Seed: 3407
Strategy: random

Training completed in 2.09 minutes.

Experiment completed successfully.
Training time: 2.09 minutes
Total time: 2.10 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\tvae_random_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\tvae_random_target.pkl


Validate the generated dataset

In [38]:
tvae_random_3407_df = pd.read_csv(
    OUTPUT_DIR / "seed_3407" / "tvae_random_target_restored.csv"
)

print("Shape:", tvae_random_3407_df.shape)

print("\nMissing values:")
print(tvae_random_3407_df.isna().sum())

print("\nTarget distribution:")
print(tvae_random_3407_df[TARGET].value_counts())

print("\nFirst five rows:")
display(tvae_random_3407_df.head())

Shape: (21328, 5)

Missing values:
nevents              3652
ndays_act            2236
ncontent                0
nforum_posts        14606
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    17979
1     3349
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,107.0,2.0,88.0,3.0,0
1,326.0,1.0,100.0,4.0,0
2,1.0,6.0,100.0,NaN,0
3,169.0,NaN,89.0,3.0,0
4,NaN,2.0,100.0,NaN,0


CTGAN, Seed 3407, From-column

In [39]:
ctgan_column_3407_result = train_generator(
    tag="ctgan_from_column_target",
    generator_class=GENERATORS["ctgan"]["class"],
    generator_kwargs=GENERATORS["ctgan"]["kwargs"],
    input_df=EXPERIMENTS["from_column_target"]["input"],
    metadata=EXPERIMENTS["from_column_target"]["metadata"],
    seed=3407,
    strategy="from_column",
)


Experiment: ctgan_from_column_target
Seed: 3407
Strategy: from_column


Gen. (-01.30) | Discrim. (-00.14): 100%|██████████| 300/300 [06:24<00:00,  1.28s/it]



Training completed in 6.60 minutes.

Experiment completed successfully.
Training time: 6.60 minutes
Total time: 6.61 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\ctgan_from_column_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\ctgan_from_column_target.pkl


validate the final CTGAN dataset

In [40]:
ctgan_column_3407_df = pd.read_csv(
    OUTPUT_DIR / "seed_3407" / "ctgan_from_column_target_restored.csv"
)

print("Shape:", ctgan_column_3407_df.shape)

print("\nMissing values:")
print(ctgan_column_3407_df.isna().sum())

print("\nTarget distribution:")
print(ctgan_column_3407_df[TARGET].value_counts())

print("\nFirst five rows:")
display(ctgan_column_3407_df.head())

Shape: (21328, 5)

Missing values:
nevents              3143
ndays_act            1949
ncontent                0
nforum_posts        13329
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    16860
1     4468
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,157.0,2.0,100,NaN,1
1,NaN,3.0,100,4.0,0
2,434.0,9.0,100,5.0,1
3,92.0,2.0,48,NaN,0
4,111.0,1.0,33,NaN,0


In [41]:
tvae_column_3407_result = train_generator(
    tag="tvae_from_column_target",
    generator_class=GENERATORS["tvae"]["class"],
    generator_kwargs=GENERATORS["tvae"]["kwargs"],
    input_df=EXPERIMENTS["from_column_target"]["input"],
    metadata=EXPERIMENTS["from_column_target"]["metadata"],
    seed=3407,
    strategy="from_column",
)


Experiment: tvae_from_column_target
Seed: 3407
Strategy: from_column

Training completed in 2.48 minutes.

Experiment completed successfully.
Training time: 2.48 minutes
Total time: 2.49 minutes
Dataset saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\tvae_from_column_target_restored.csv
Model saved: d:\student_engagement_twin\artifacts\robustness\seed_3407\tvae_from_column_target.pkl


Final dataset validation

In [42]:
tvae_column_3407_df = pd.read_csv(
    OUTPUT_DIR / "seed_3407" / "tvae_from_column_target_restored.csv"
)

print("Shape:", tvae_column_3407_df.shape)

print("\nMissing values:")
print(tvae_column_3407_df.isna().sum())

print("\nTarget distribution:")
print(tvae_column_3407_df[TARGET].value_counts())

print("\nFirst five rows:")
display(tvae_column_3407_df.head())

Shape: (21328, 5)

Missing values:
nevents              3387
ndays_act            1757
ncontent                0
nforum_posts        14547
Engagement_Level        0
dtype: int64

Target distribution:
Engagement_Level
0    16971
1     4357
Name: count, dtype: int64

First five rows:


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,195.0,3.0,65,1.0,0
1,1256.0,3.0,100,14.0,1
2,40.0,16.0,100,2.0,1
3,2199.0,8.0,100,NaN,1
4,402.0,15.0,100,NaN,1
